*Previous notebook: L1 and L2 regularization and their geometric intuition. — This notebook: dropout and other regularization methods.*

# Dropout

The dropout regularization eliminates some neurons/weights on each iteration based on a probability. For a given layer (for example layer 3), dropout is implemented as follows:

```python
keep_prob = 0.8

d3 = np.random.rand(a3.shape[0], a3.shape[1]) < keep_prob
a3 = np.multiply(a3, d3)
a3 = a3 / keep_prob
```

The mask d3 determines which neurons are kept and which are dropped. After applying it, the activations are divided by keep_prob to compensate for the fact that some activations were set to zero. Without this scaling, the output of the layer would systematically shrink during training, creating a mismatch between training and test behavior.

Dropout can have different keep_prob per layer. The input layer dropout has to be near 1 (or 1 - no dropout) because you don't want to eliminate a lot of features. At test time we don't use dropout. If you implement dropout at test time - it would add noise to predictions.

One downside of dropout is that it makes the cost function less smooth during training. The value of the loss can fluctuate from iteration to iteration, which makes debugging more difficult. A common strategy to verify that everything is working correctly is to temporarily disable dropout (set all keep_prob to 1) and check that the cost decreases monotonically. Once confirmed, dropout can be re-enabled.

---

### Why Dropout Works

Intuitively, dropout works because it prevents neurons from co-adapting too strongly. Since any neuron can be randomly removed at any time, each neuron is forced to learn features that are useful in combination with many different subsets of other neurons. This leads to a more redundant and distributed representation, which typically generalizes better to unseen data.

Another way to think about it is that dropout implicitly trains a large ensemble of subnetworks that share parameters. The final model can be interpreted as an approximation of averaging the predictions of all these subnetworks.

Interestingly, dropout has a regularizing effect similar to L2 regularization, even though the mechanism is completely different. Instead of shrinking weights directly, it injects noise into the network during training, which discourages overly complex co-adaptations.

# Other Regularization Methods

Beyond L1, L2 regularization and dropout, there are several other practical techniques used to reduce overfitting and improve generalization.

One of the most common is **data augmentation**, especially in computer vision tasks. The idea is to artificially increase the size of the training set by applying label-preserving transformations to existing examples. For instance, images can be flipped horizontally to create new valid samples that still represent the same class. Similarly, random rotations, translations, and small distortions can be applied to images to generate additional training data. In tasks such as optical character recognition (OCR), introducing random rotations and elastic distortions of digits or letters can significantly improve robustness.

Although these artificially generated samples are not as informative as truly independent real-world data, they still help reduce overfitting because they expose the model to a wider variety of input variations. In practice, data augmentation acts as a powerful implicit regularizer by making the model less sensitive to small changes in the input.

Another important technique is **early stopping**. During training, both the training set error and the development set error are monitored over iterations. Typically, the training error keeps decreasing, while the development error eventually reaches a minimum and then starts increasing due to overfitting. Early stopping consists of selecting the parameters at the point where the development error is lowest, even if the training error has not fully converged.

This method is effective because it prevents the model from continuing to fit noise in the training data. However, Andrew Ng often notes that L2 regularization is generally preferred over early stopping. The reason is conceptual: early stopping simultaneously tries to minimize training error while also limiting overfitting, which can be seen as coupling two objectives that ideally should be handled separately (this relates to the idea of orthogonalization). L2 regularization, in contrast, cleanly separates these goals by explicitly adding a penalty term to the cost function.

That said, early stopping has a practical advantage: it does not require tuning an additional hyperparameter such as λ in L2 regularization, which can simplify the model selection process.

Finally, another powerful approach is the use of **model ensembles**. The idea is simple: train multiple independent neural networks and average their predictions at test time. This usually leads to improved performance because different models tend to make different errors, and averaging helps reduce variance. In practice, ensembles can improve accuracy by a few percentage points, which can be very significant in competitive settings.

A practical variant of this idea is to take multiple “snapshots” of a single neural network during training and combine them as an ensemble. This allows one to approximate the benefits of ensembling without training completely separate models from scratch. 